In [18]:
import requests
from bs4 import BeautifulSoup

url = "https://globalsummitguide.com/master-mountain-list"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

raw_names = []
mountains_list = soup.find_all('td', class_=None)

# find names in table (in span or a)
for td in mountains_list:
    a_tag = td.find('a')
    span_tag = td.find('span', class_='mtn-plain')

    if a_tag:
        raw_names.append(a_tag.get_text(strip=True))
    elif span_tag:
        raw_names.append(span_tag.get_text(strip=True))

# сlean from text in brackets
clean_names = []
for name in raw_names:
    if "(" in name:
        name = name.split("(")[0].strip()
    if name and name not in clean_names:
        clean_names.append(name)

print(f"Amount of collected mountains: {len(clean_names)}")
print(clean_names[:5])

Amount of collected mountains: 498
['Mount Everest', 'K2', 'Kangchenjunga', 'Lhotse', 'Makalu']


In [19]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

openapi_key=os.getenv("AI_API")



In [20]:
import random

# pool of structural styles to force syntactic diversity
style_pool = [
    "use passive voice at least once",
    "write one sentence as a direct quote from a person",
    "make one sentence look like an old historical record or archive text",
    "include other geographic features like rivers, valleys, or cities in the same text",
    "use complex punctuation (dashes, semicolons, or parenthetical remarks)",
    "start at least one sentence with a subordinate clause (e.g., 'although...', 'while...')",
    "make one sentence look like a technical scientific report about geology",
    "ensure the mountain name appears in the exact middle or end of the text, not just the beginning",
]

# list of corporate environments for negatives
negative_contexts = [
    "gaming/video game titles",
    "luxury watch and fountain pen manufacturing",
    "venture capital, asset management, and investment funds",
    "streetwear, outdoor clothing lines, and fashion brands",
    "cloud computing, infrastructure software, and database tech startups",
    "local artisanal cafes, bistros, and roasteries",
]

def get_dynamic_prompt(mountain_batch: list, positive: bool):
    mountains_str = ", ".join(mountain_batch)

    # pick random constraints for this specific batch loop
    selected_styles = random.sample(style_pool, k=4)
    selected_negatives = random.sample(negative_contexts, k=4)

    selected_sentence = "2 positive sentences (where it is a real mountain)" if positive else "2 hard negative sentences (where it is NOT a mountain)"

    # assemble the ultimate prompt
    prompt = f"""
    act as an expert computational linguist and gold-standard ner data annotator.
    your task is to generate a highly diverse, non-templated dataset for named entity recognition (class: MOUNTAIN).

    target entities for this batch: [{mountains_str}]

    generation requirements:
    1. for each entity, generate exactly {selected_sentence}.
    2. structural diversity constraints for this batch (apply them across your answers):
       - {selected_styles[0]}
       - {selected_styles[1]}
    3. negative context constraints for this batch:
       - use these specific domains for corporate/product naming traps: {selected_negatives[0]} or {selected_negatives[1]}.
    4. linguistic rule: strictly avoid generic clichés like "the view from the base camp is breathtaking", "reached the summit at dawn", or simply "[mountain] logistics". make sentences natural, complex, and human-written.
    5. you are absolutely forbidden to write anything other than the json of the response, neither at the beginning nor at the end of the response, you start and end your response exclusively with a dataset without any unnecessary phrases.

    return only a valid json object with a single key "data" that contains the array.
    do not wrap the response in markdown code fences.
    expected json structure:
    {{
      "data": [
        {{
          "text": "sentence content here",
          "entities": [{{"start": int, "end": int, "label": "MOUNTAIN"}}]
        }}
      ]
    }}
    """
    return prompt

In [26]:
import json
import google.generativeai as gemini

gemini.configure(api_key=openapi_key)


def generate_dataset_from_chatgpt(mountain_batch, positive):
    mountains_str = ", ".join(mountain_batch)
    prompt = get_dynamic_prompt(mountains_str, positive)

    try:
        model = gemini.GenerativeModel(
            "gemini-3.5-flash",
            generation_config={"response_mime_type": "application/json"},
        )
        response = model.generate_content(prompt)
        raw_json = json.loads(response.text)

        # safely extract the target list from our new data root key
        if "data" in raw_json and isinstance(raw_json["data"], list):
            return raw_json["data"]

        # backup recovery path just in case the key name variance happens
        for key in raw_json.keys():
            if isinstance(raw_json[key], list):
                return raw_json[key]

        print(f"expected data list key missing for batch: {mountain_batch}")
        return []

    except Exception as e:
        # no more parsing crashes, handles rate limits or network drops gracefully
        print(f"failed batch {mountain_batch}: {e}")
        return []

In [ ]:
dataset_positive = []
dataset_negative = []

for i in range(0, 355, 6):
    answer_positive=generate_dataset_from_chatgpt(clean_names[i:i+6], True)
    dataset_positive.extend(answer_positive)
    answer_negative=generate_dataset_from_chatgpt(clean_names[i:i+6], False)
    dataset_negative.extend(answer_negative)

In [28]:
import json

with open("dataset_positive.json", "w", encoding="utf-8") as file:
    json.dump(dataset_positive, file, ensure_ascii=False, indent=2)

with open("dataset_negative.json", "w", encoding="utf-8") as file:
    json.dump(dataset_negative, file, ensure_ascii=False, indent=2)

In [29]:
import json
import re

with open("dataset_positive.json", "r", encoding="utf-8") as f:
    positive_dataset = json.load(f)

with open("dataset_negative.json", "r", encoding="utf-8") as f:
    negative_dataset = json.load(f)

clean_names.sort(key=len, reverse=True)

fixed_dataset = []

for item in positive_dataset:
    text = item["text"]
    new_entities = []

    for mtn in clean_names:
        pattern = r"\b" + re.escape(mtn) + r"\b"

        for match in re.finditer(pattern, text):
            start_idx = match.start()
            end_idx = match.end()
            overlap = any(
                start_idx < e["end"] and end_idx > e["start"]
                for e in new_entities
            )

            if not overlap:
                new_entities.append(
                    {"start": start_idx, "end": end_idx, "label": "MOUNTAIN"}
                )

    if new_entities:
        new_entities.sort(key=lambda x: x["start"])
        item["entities"] = new_entities
        fixed_dataset.append(item)
    else:
        print(f"skipping positive item without active match: {text}")

for item in negative_dataset:
    item["entities"] = []
    fixed_dataset.append(item)

with open("final_train_dataset.json", "w", encoding="utf-8") as f:
    json.dump(fixed_dataset, f, ensure_ascii=False, indent=2)

print(f"Total clean records saved: {len(fixed_dataset)}")
print(f" - Positive samples: {len(fixed_dataset) - len(negative_dataset)}")
print(f" - Hard Negative samples: {len(negative_dataset)}")

skipping positive item without active match: In the archives of the Alpine Club, the 1864 ascent of the Aiguille d'Argentière by Edward Whymper is documented as a pivotal moment in the golden age of alpinism.
Total clean records saved: 74
 - Positive samples: 38
 - Hard Negative samples: 36


In [30]:
# load current file to check the slices
with open("dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

print("--- validation report ---")
for i, item in enumerate(dataset):
    text = item["text"]
    entities = item["entities"]

    if entities:
        start = entities[0]["start"]
        end = entities[0]["end"]
        # pulling the actual string that lives under these indices
        actual_substring = text[start:end]

        if actual_substring not in clean_names:
            print(f"line {i} | indices: {start}:{end} | actual text inside: '{actual_substring}'")
        else:
            print(f"{actual_substring}")

--- validation report ---
Mount Everest
Mount Everest
K2
K2
Kangchenjunga
Kangchenjunga
Lhotse
Lhotse
Makalu
Makalu
Cho Oyu
Cho Oyu
Dhaulagiri I
Dhaulagiri I
Manaslu
Manaslu
Nanga Parbat
Nanga Parbat
Annapurna I
Annapurna I
Gasherbrum I
Gasherbrum I
Broad Peak
Broad Peak
Gasherbrum II
Gasherbrum II
Shishapangma
Shishapangma
Aconcagua
Aconcagua
Denali
Denali
Mount Kilimanjaro
Mount Kilimanjaro
Mount Elbrus
Mount Elbrus
Island Peak
Island Peak
Island Peak
Island Peak
Mera Peak
Mera Peak
Mera Peak
Mera Peak
Lobuche East
Lobuche East
Lobuche East
Lobuche East
Baruntse
Baruntse
Baruntse
Baruntse
Mount Kailash
Mount Kailash
Mount Kailash
Mount Kailash
Mount Fuji
Mount Fuji
Mount Fuji
Mount Fuji
Mount Kinabalu
Mount Kinabalu
Mount Rinjani
Mount Rinjani
Mount Agung
Mount Agung
Mount Apo
Mount Apo
Mount Ararat
Mount Ararat
Mount Damavand
Mount Damavand
Mount Kazbek
Mount Kazbek
Mount Kazbek
Mount Kazbek
Khan Tengri
Khan Tengri
Khan Tengri
Khan Tengri
Jengish Chokusu
Jengish Chokusu
Jengish Chok

In [27]:
for i in range(400, 500, 1):
    print(clean_names[i], end=", ")

Pharchamo, Abi Gamin, Mana Peak, Nanda Kot, Mrigthuni, Trisul II, Deo Tibba, Rakaposhi, Ultar Sar, Passu Sar, Tajumulco, Parícutin, Condoriri, Tupungato, Baruntse, Jungfrau, Alpamayo, Cotopaxi, Illimani, Fitz Roy, La Meije, Mulhacén, Trisul I, Shivling, Cholatse, Kyajo Ri, Gokyo Ri, El Misti, Coropuna, Yerupajá, Acotango, Piz Palü, Piz Buin, Alphubel, Tetnuldi, Dykh-Tau, Ganesh I, Kangtega, Dunagiri, Indrasan, Antisana, Ancohuma, Mururata, Tronador, Cho Oyu, Manaslu, Civetta, Lyskamm, Taboche, Illampu, Pelvoux, Bishorn, Mytikas, Hoverla, Shkhara, Hardeol, Spantik, Latok I, Cayambe, Huandoy, Sabalan, Alamkuh, Lhotse, Makalu, Denali, Nuptse, Pumori, Sajama, Ortler, Pollux, Castor, Posets, Veleta, Musala, Midžor, Negoiu, Saipal, Numbur, Noshaq, Batian, Nelion, Mafadi, Tacaná, Sangay, Tunari, Osorno, Llaima, Eiger, Kamet, Ushba, Altar, Lanín, Rysy, Dom, Nun, Kun, Api, K2, 

IndexError: list index out of range

In [ ]:
окей дивись попередні спроби генерувати щось через код і апі провалились твоя задача для створення ner датасета, ось промпти  def get_dynamic_prompt(mountains_str, positive):

    style = random.choice(style_pool)

    if positive:
      task = f"""
      You are generating training data for a Named Entity Recognition (NER) dataset.

Entity:
"{mountains_str}"

Task:

Generate EXACTLY 2 different natural English sentences.

Rules:

- Use the entity EXACTLY as written.
- Do NOT modify, translate, shorten, or replace its name.
- Every sentence MUST contain the entity exactly once.
- The entity MUST refer to the real mountain.
- The mountain should be the only mountain mentioned in the sentence.
- Do NOT mention any other mountain names.
- Do NOT use placeholders.
- Do NOT use HTML tags.
- Do NOT use markdown.

Write natural human-written sentences.

Vary the context across examples, such as:

- geology
- ecology
- climate
- biodiversity
- cartography
- environmental science
- tourism
- local culture
- history
- scientific research

Avoid repetitive summit-climbing clichés.

Do not make uncertain factual claims about rankings or historical events.

Return ONLY valid JSON.

{{
  "data": [
    {{
      "text": "sentence content here",
      "is_mountain": true
    }},
    {{
      "text": "another sentence",
      "is_mountain": true
    }}
  ]
}}
      """

    else:
      task = f"""
      You are generating hard negative examples for a Named Entity Recognition (NER) dataset.

Entity:
"{mountains_str}"

Task:

Generate EXACTLY 2 different natural English sentences.
The ONLY entity name allowed in the generated sentences is:

"{mountains_str}"

Every sentence MUST contain this exact entity.

Using any other mountain name makes the answer incorrect.

Rules:

- Use the entity EXACTLY as written.
- Do NOT modify, translate, shorten, or replace its name.
- Every sentence MUST contain the entity exactly once.
- Do NOT introduce any additional named entities with the same name.
- The entity MUST NOT refer to a mountain or any geographical feature.

Instead, use the entity naturally as the name of a:

- company
- organization
- startup
- software
- product
- service
- investment fund
- university
- conference
- research institute
- mobile app
- technology
- event
- foundation
- media outlet

IMPORTANT:

When the entity starts with "Mount", omit the word "Mount" and use only the distinctive part of the name.

Example:
Mount Everest → Everest
Mount Elbrus → Elbrus

Do not write "Mount Everest" when generating negative examples.

The surrounding context must clearly indicate that the entity is NOT a mountain.

Do NOT use mountain-related vocabulary.


Forbidden words include:

mountain
peak
summit
ridge
glacier
climber
climbing
mountaineer
expedition
trail
trek
hiking
snow
ice
altitude
alpine
valley
slope
base camp
Himalaya
Karakoram

Do NOT describe landscapes.

Do NOT describe tourism.

Do NOT describe outdoor activities.

Most sentences should NOT explicitly contain words such as:

company
brand
product
startup
software

Instead, let the reader infer the meaning naturally.

Examples of good negatives:

Everest announced its quarterly earnings yesterday.

K2 released a major software update.

Denali opened three new offices across Europe.

Makalu received regulatory approval for its new platform.

Return ONLY valid JSON.

{{
  "data": [
    {{
      "text": "sentence content here",
      "is_mountain": false
    }},
    {{
      "text": "another sentence",
      "is_mountain": false
    }}
  ]
}}
  """

    return f"""
You are generating a Named Entity Recognition dataset.

Task:

{task}

Rules:

- {style}
- Output ONLY JSON.
- No markdown.
- No explanation.
- Preserve exact entity spelling.

Return:

{{
 "data":[
   {{
      "text":"sentence content here",
      "is_mountain":true/false
   }}
 ]
}} зроби для кожної назви відповідні але окремі файли json негативні окремо позитивні окремо 